# Proyecto 4: Motor de Trading Algorítmico

**Objetivo**: Construir un motor de trading algorítmico usando CLOS para modelar instrumentos financieros, órdenes y estrategias, con backtesting en Python.

**Features demostradas**:
- `defclass` y `defmethod` para modelado de dominio financiero
- `defgeneric` con especialización por clase
- Hash tables para book de órdenes
- `py:call-kw` para argumentos keyword en funciones Python
- NumPy para cálculos de métricas financieras (Sharpe ratio, drawdown)
- Visualización de equity curve con matplotlib
- Callbacks Lisp→Python para estrategias de trading

## 1. Inicialización

In [ ]:
(load "lib/numpy-fast.lp")
(py:import "numpy")
(py:import "matplotlib.pyplot")

## 2. Modelo de Dominio con CLOS

Definimos las clases fundamentales del sistema de trading.

In [ ]:
;; Instrumento financiero
(defclass instrument ()
  ((symbol :initarg :symbol :reader symbol)
   (name :initarg :name :reader name)
   (price :initarg :price :accessor price :initform 0.0)
   (history :initform nil :accessor history)))

;; Orden
(defclass order ()
  ((instrument :initarg :instrument :reader order-instrument)
   (side :initarg :side :reader side)        ;; :buy or :sell
   (quantity :initarg :quantity :reader quantity)
   (limit-price :initarg :limit-price :reader limit-price :initform nil)
   (timestamp :initarg :timestamp :reader timestamp)
   (status :initform :pending :accessor status)))

;; Portfolio
(defclass portfolio ()
  ((cash :initarg :cash :accessor cash :initform 100000.0)
   (positions :initform (make-hash-table :test 'equal) :accessor positions)
   (trades :initform nil :accessor trades)))

;; Estrategia base
(defclass strategy ()
  ((name :initarg :name :reader strategy-name)
   (parameters :initarg :parameters :accessor parameters :initform nil)))

(defgeneric execute-strategy (strategy instrument history portfolio)
  (:documentation "Execute strategy and return list of orders"))

## 3. Operaciones del Portfolio

Funciones para gestionar posiciones y ejecutar órdenes.

In [ ]:
(defun get-position (portfolio instrument)
  (gethash (symbol instrument) (positions portfolio) 0))

(defun (setf get-position) (value portfolio instrument)
  (setf (gethash (symbol instrument) (positions portfolio)) value))

(defun execute-order (portfolio order)
  (let* ((inst (order-instrument order))
         (qty (quantity order))
         (px (price inst))
         (cost (* qty px)))
    (cond
      ((eq (side order) :buy)
       (when (>= (cash portfolio) cost)
         (decf (cash portfolio) cost)
         (incf (get-position portfolio inst) qty)
         (setf (status order) :filled)
         (push order (trades portfolio))
         t))
      ((eq (side order) :sell)
       (when (>= (get-position portfolio inst) qty)
         (incf (cash portfolio) cost)
         (decf (get-position portfolio inst) qty)
         (setf (status order) :filled)
         (push order (trades portfolio))
         t))
      (t (setf (status order) :rejected) nil))))

(defun portfolio-value (portfolio)
  (let ((total (cash portfolio)))
    (maphash (lambda (sym qty)
               (declare (ignore sym))
               (incf total (* qty (price (make-instance 'instrument :symbol sym)))))
             (positions portfolio))
    total))

## 4. Estrategias de Trading

Implementamos dos estrategias: Media Móvil y Buy & Hold.

In [ ]:
;; Estrategia: Cruce de Medias Móviles
(defclass ma-cross-strategy (strategy) ())

(defmethod execute-strategy ((s ma-cross-strategy) instrument history portfolio)
  (let* ((params (parameters s))
         (short-window (or (cdr (assoc :short-window params)) 5))
         (long-window (or (cdr (assoc :long-window params)) 20))
         (prices (mapcar (lambda (h) (price h)) history))
         (n (length prices)))
    (when (< n long-window) (return-from execute-strategy nil))
    (let* ((short-ma (/ (reduce #'+ (subseq prices (- n short-window) n)) short-window))
           (long-ma (/ (reduce #'+ (subseq prices (- n long-window) n)) long-window))
           (prev-prices (butlast prices))
           (prev-short (/ (reduce #'+ (subseq prev-prices (- (length prev-prices) short-window))) short-window))
           (prev-long (/ (reduce #'+ (subseq prev-prices (- (length prev-prices) long-window))) long-window))
           (position (get-position portfolio instrument)))
      (format t "  Short MA: ~,2F, Long MA: ~,2F~%" short-ma long-ma)
      ;; Señal de compra: cruce alcista
      (when (and (<= prev-short prev-long) (> short-ma long-ma) (= position 0))
        (format t "  >> SEÑAL COMPRA <<~%")
        (return-from execute-strategy
          (list (make-instance 'order
                  :instrument instrument :side :buy
                  :quantity (floor (/ (cash portfolio) 2) (price instrument))
                  :timestamp (length history)))))
      ;; Señal de venta: cruce bajista
      (when (and (>= prev-short prev-long) (< short-ma long-ma) (> position 0))
        (format t "  >> SEÑAL VENTA <<~%")
        (return-from execute-strategy
          (list (make-instance 'order
                  :instrument instrument :side :sell
                  :quantity position
                  :timestamp (length history))))))
    nil))

;; Estrategia: Buy & Hold
(defclass buy-hold-strategy (strategy) ())

(defmethod execute-strategy ((s buy-hold-strategy) instrument history portfolio)
  (when (= (length history) 1)
    (list (make-instance 'order
            :instrument instrument :side :buy
            :quantity (floor (cash portfolio) (price instrument))
            :timestamp 0))))

## 5. Generación de Datos de Mercado

Creamos precios sintéticos con movimiento browniano geométrico.

In [ ]:
(py:run "
import numpy as np

np.random.seed(42)

def generate_prices(n=252, s0=100.0, mu=0.1, sigma=0.25):
    dt = 1/252
    returns = np.random.normal(mu*dt, sigma*np.sqrt(dt), n)
    prices = s0 * np.exp(np.cumsum(returns))
    return prices.tolist()

precios_activo = generate_prices(252, 100.0, 0.08, 0.30)
print(f'Generados {len(precios_activo)} días de precios')
print(f'Precio inicial: ${precios_activo[0]:.2f}')
print(f'Precio final:   ${precios_activo[-1]:.2f}')
print(f'Retorno total:  {((precios_activo[-1]/precios_activo[0] - 1) * 100):.2f}%')
")

## 6. Backtesting Engine

Ejecutamos el backtest día por día, aplicando las estrategias.

In [ ]:
;; Crear instrumento y portfolio
(defvar *activo* (make-instance 'instrument :symbol "ACT" :name "Activo Sintético"))
(defvar *portfolio* (make-instance 'portfolio :cash 100000.0))
(defvar *estrategia* (make-instance 'ma-cross-strategy
                       :name "MA-Cross"
                       :parameters '((:short-window . 5) (:long-window . 20))))

;; Ejecutar backtest
(py:run "equity_curve = []")
(let ((prices @precios_activo))
  (dolist (p prices)
    (setf (price *activo*) p)
    (push (make-instance 'instrument
            :symbol "ACT" :price p)
          (history *activo*))
    (let ((orders (execute-strategy *estrategia* *activo* (history *activo*) *portfolio*)))
      (dolist (order orders)
        (execute-order *portfolio* order)))
    ;; Track equity
    (let ((value (portfolio-value *portfolio*)))
      (py:run (format nil "equity_curve.append(~F)" value)))))

(format t "~%=== Resultados Backtest ===~%")
(format t "Cash final: $~,2F~%" (cash *portfolio*))
(format t "Trades ejecutados: ~D~%" (length (trades *portfolio*)))
(format t "Valor portfolio: $~,2F~%" (portfolio-value *portfolio*))

## 7. Métricas de Rendimiento con NumPy

Calculamos Sharpe ratio, drawdown máximo y retorno anualizado.

In [ ]:
(py:run "
import numpy as np

equity = np.array(equity_curve)
returns = np.diff(equity) / equity[:-1]

# Métricas
total_return = (equity[-1] / equity[0] - 1) * 100
annual_return = (equity[-1] / equity[0]) ** (252 / len(equity)) - 1
sharpe = np.sqrt(252) * np.mean(returns) / np.std(returns) if np.std(returns) > 0 else 0

# Maximum drawdown
peak = np.maximum.accumulate(equity)
drawdown = (equity - peak) / peak
max_dd = np.min(drawdown) * 100

# Volatilidad
volatility = np.std(returns) * np.sqrt(252) * 100

print('=== Métricas de Rendimiento ===')
print(f'Retorno Total:      {total_return:.2f}%')
print(f'Retorno Anualizado:  {annual_return*100:.2f}%')
print(f'Sharpe Ratio:        {sharpe:.3f}')
print(f'Volatilidad Anual:   {volatility:.2f}%')
print(f'Max Drawdown:        {max_dd:.2f}%')
print(f'Valor Final:         ${equity[-1]:.2f}')
")

## 8. Visualización — Equity Curve y Drawdown

In [ ]:
(py:run "
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Equity curve
axes[0].plot(equity, 'b-', linewidth=1.5)
axes[0].axhline(y=100000, color='gray', linestyle='--', alpha=0.5, label='Inicial $100K')
axes[0].set_ylabel('Equity ($)')
axes[0].set_title('Equity Curve - Estrategia MA Cross')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Drawdown
axes[1].fill_between(range(len(drawdown)), 0, drawdown*100, color='red', alpha=0.3)
axes[1].plot(drawdown*100, 'r-', linewidth=1)
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_title('Drawdown')
axes[1].grid(True, alpha=0.3)

# Returns diarios
axes[2].bar(range(len(returns)), returns*100, width=1, alpha=0.7)
axes[2].set_ylabel('Retorno Diario (%)')
axes[2].set_xlabel('Día')
axes[2].set_title('Retornos Diarios')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('trading-results.png', dpi=100)
plt.show()
")

## 9. Comparación de Estrategias desde Lisp

Ejecutamos múltiples configuraciones de la estrategia y comparamos.

In [ ]:
(defun backtest-strategy (short-w long-w)
  (let* ((port (make-instance 'portfolio :cash 100000.0))
         (act (make-instance 'instrument :symbol "ACT"))
         (strat (make-instance 'ma-cross-strategy
                  :name "MA"
                  :parameters `((:short-window . ,short-w) (:long-window . ,long-w))))
         (prices @precios_activo))
    (dolist (p prices)
      (setf (price act) p)
      (push (make-instance 'instrument :symbol "ACT" :price p) (history act))
      (let ((orders (execute-strategy strat act (history act) port)))
        (dolist (o orders) (execute-order port o))))
    (list :short short-w :long long-w
          :trades (length (trades port))
          :final-value (portfolio-value port))))

(format t "=== Comparación de Parámetros ===~%")
(format t "~8a ~8a ~8a ~15a~%" "Short" "Long" "Trades" "Valor Final")
(format t "~%~{~%")
(dolist (config '((3 . 10) (5 . 20) (10 . 30) (3 . 15) (7 . 25)))
  (let* ((result (backtest-strategy (car config) (cdr config))))
    (format t "~8D ~8D ~8D $~,2F~%"
            (getf result :short) (getf result :long)
            (getf result :trades) (getf result :final-value))))

## 10. Ejercicios Propuestos

1. Implementar una estrategia de reversión a la media con bandas de Bollinger
2. Agregar `:limit-order` con precio límite y lógica de ejecución condicional
3. Implementar gestión de riesgo: stop-loss, take-profit, position sizing
4. Backtest con múltiples activos simultáneos y correlación
5. Exportar resultados a CSV para análisis externo con pandas